# EfficientNetV2-B3 — training on the original data

Trains the mask classifier (**Classifier-Orig-Mask**) on the [Face Mask 12k dataset](https://www.kaggle.com/datasets/ashishjangra27/face-mask-12k-images-dataset): frozen ImageNet backbone + `Dense(2, softmax)` head, 64×64 input, batch 128, blur augmentation, 6 epochs. Ran on Kaggle GPUs.

CLI equivalent: `python -m src.train --model efficientnet`.

In [8]:
from pathlib import Path

import keras
from keras.applications import EfficientNetV2B3
# from keras.applications.efficientnet_v2 import preprocess_input
from keras import Sequential
from keras.layers import Flatten, Dense
from keras.preprocessing.image import ImageDataGenerator

import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [9]:
keras.utils.set_random_seed(0)
tf.random.set_seed(0)
np.random.seed(0)

In [10]:
train_dir = '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Train'
test_dir = '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Test'
val_dir = '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Validation'

In [11]:
batch_size = 128
input_dim = 64


def preprocess(x):
    blur_kernel = np.random.randint(0, 4) * 2 + 1
    x = cv2.GaussianBlur(x, (blur_kernel, blur_kernel), 0)
    return x


# Rescaling is done with include_preprocessing=True parameter of EfficientNetV2M

train_datagen = ImageDataGenerator(horizontal_flip=True, zoom_range=0.2, shear_range=0.2, brightness_range=[0.6, 1.0], preprocessing_function=preprocess)
train_generator = train_datagen.flow_from_directory(directory=train_dir,
                                                    target_size=(input_dim, input_dim),
                                                    class_mode='categorical',
                                                    batch_size=batch_size)

val_datagen = ImageDataGenerator()
val_generator = train_datagen.flow_from_directory(directory=val_dir,
                                                  target_size=(input_dim, input_dim),
                                                  class_mode='categorical',
                                                  batch_size=batch_size)

test_datagen = ImageDataGenerator()
test_generator = train_datagen.flow_from_directory(directory=val_dir,
                                                   target_size=(input_dim, input_dim),
                                                   class_mode='categorical',
                                                   batch_size=batch_size)

Found 10000 images belonging to 2 classes.
Found 800 images belonging to 2 classes.
Found 800 images belonging to 2 classes.


In [12]:
efficient_net = EfficientNetV2B3(weights='imagenet', include_top=False, input_shape=(input_dim, input_dim, 3), include_preprocessing=True)

for layer in efficient_net.layers:
    layer.trainable = False
    
model = Sequential()
model.add(efficient_net)
model.add(Flatten())
model.add(Dense(2, activation='softmax'))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetv2-b3 (Functio  (None, 2, 2, 1536)        12930622  
 nal)                                                            
                                                                 
 flatten_1 (Flatten)         (None, 6144)              0         
                                                                 
 dense_1 (Dense)             (None, 2)                 12290     
                                                                 
Total params: 12942912 (49.37 MB)
Trainable params: 12290 (48.01 KB)
Non-trainable params: 12930622 (49.33 MB)
_________________________________________________________________


In [13]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [14]:
history = model.fit(train_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=6,
                    validation_data=val_generator,
                    validation_steps=len(val_generator))

Epoch 1/6
79/79 [==============================] - 91s 977ms/step - loss: 0.0828 - accuracy: 0.9700 - val_loss: 0.0334 - val_accuracy: 0.9875
Epoch 2/6
79/79 [==============================] - 67s 846ms/step - loss: 0.0387 - accuracy: 0.9874 - val_loss: 0.0220 - val_accuracy: 0.9962
Epoch 3/6
79/79 [==============================] - 67s 849ms/step - loss: 0.0297 - accuracy: 0.9905 - val_loss: 0.0257 - val_accuracy: 0.9900
Epoch 4/6
79/79 [==============================] - 73s 922ms/step - loss: 0.0273 - accuracy: 0.9908 - val_loss: 0.0271 - val_accuracy: 0.9962
Epoch 5/6
79/79 [==============================] - 67s 851ms/step - loss: 0.0267 - accuracy: 0.9919 - val_loss: 0.0148 - val_accuracy: 0.9950
Epoch 6/6
79/79 [==============================] - 71s 901ms/step - loss: 0.0244 - accuracy: 0.9926 - val_loss: 0.0061 - val_accuracy: 1.0000


In [15]:
model.evaluate(test_generator)

7/7 [==============================] - 5s 662ms/step - loss: 0.0151 - accuracy: 0.9950


[0.01509585976600647, 0.9950000047683716]

In [16]:
model.save('efficientnetv2_b3_orig6_softmax.h5')

/opt/conda/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [17]:
import json

with open('history_orig6_softmax.json', 'w') as f:
    json.dump(history.history, f, indent=4)